## Overview
Programs are tools for reducing future healthcare costs. As the number of offered programs increases, it becomes more challenging to determine the program that will be the best fit for members in need of support. Here, we are considering 89 programs across 2.8 million members. A program recommendation engine was built and deployed in Snowflake to algorithmically surface program recommendations in the Cohort Finder Application.

This recommendation engine uses **collaborative filtering** to recommend programs to individuals based on what programs have been taken. Members who have taken similar programs will be recommended other programs that are commonly taken by these members.

Because programs have varying eligibility criteria, a **hybrid recommendation engine** is used. This approach uses member features and program features as weightings in the recommendation.


In [22]:
# Import Python packages
import snowflake.snowpark.types as T
import snowflake.snowpark.functions as F
from snowflake.snowpark import Session
from snowflake.ml.feature_store import CreationMode, FeatureStore, feature_view
from snowflake.ml.feature_store.entity import Entity
from snowflake.ml.registry import Registry
from lightfm.data import Dataset
from lightfm import LightFM
import joblib
import cachetools
import sys
import pandas as pd
import numpy as np
import json
import warnings
import os
import configparser
warnings.filterwarnings("ignore")

# Import Snowflake modules

# Snowpark ML
# Import Python packages

# Import Snowflake modules
import snowflake.ml.modeling.preprocessing as snowml

In [6]:
from snowflake.snowpark import Session
from snowflake.snowpark.context import get_active_session


def create_local_session(config_file="config.env", connection_name="connections.my_example_connection"):
    """
    Create a Snowpark session from a local config file for development/testing.

    Parameters
    ----------
    config_file : str, optional
        Path to the config file containing connection parameters.
    connection_name : str, optional
        Name of the connection section in the config file.

    Returns
    -------
    snowflake.snowpark.Session
        A Snowpark session for local development.
    """

    # Read the config file
    config = configparser.ConfigParser()
    config.read(config_file)

    if connection_name not in config:
        raise ValueError(
            f"Connection '{connection_name}' not found in {config_file}")

    # Extract connection parameters and strip quotes - simplified approach
    conn_params = {
        "account": config[connection_name]["account"].strip('"'),
        "user": config[connection_name]["user"].strip('"'),
        "role": config[connection_name]["role"].strip('"'),
        "warehouse": config[connection_name]["warehouse"].strip('"'),
        "database": config[connection_name]["database"].strip('"'),
        "schema": config[connection_name]["schema"].strip('"')
    }

    # Add password if it exists - keep it simple
    if "password" in config[connection_name]:
        conn_params["password"] = config[connection_name]["password"].strip(
            '"')

    # Create session directly without complex auth logic
    session = Session.builder.configs(conn_params).create()
    print(f"✅ Local Snowpark session created successfully")
    print(f"   Account: {conn_params['account']}")
    print(f"   User: {conn_params['user']}")
    print(f"   Role: {conn_params['role']}")
    print(f"   Warehouse: {conn_params['warehouse']}")
    print(f"   Database: {conn_params['database']}")
    print(f"   Schema: {conn_params['schema']}")

    return session

In [8]:
from snowflake.snowpark.context import get_active_session
session = create_local_session()

# get current solution prefix from warehouse name
solution_prefix = session.get_current_warehouse().strip('"').split('_BI_WH')[0]


# Print the current role, warehouse, and database/schema
print(f"role: {session.get_current_role()} | WH: {session.get_current_warehouse()} | DB.SCHEMA: {session.get_fully_qualified_current_schema()}")

✅ Local Snowpark session created successfully
   Account: zqb38977.us-east-1
   User: kaitlyn
   Role: SNOWFLAKE_INTELLIGENCE_ADMIN_RL
   Warehouse: CORTEX_ANALYST_WH
   Database: CORTEX_ANALYTICS
   Schema: PUBLIC
role: "SNOWFLAKE_INTELLIGENCE_ADMIN_RL" | WH: "CORTEX_ANALYST_WH" | DB.SCHEMA: "CORTEX_ANALYTICS"."PUBLIC"


In [ ]:
##Set the Session Context
session.use_database("COHORT_BUILDER")
session.use_schema("RAW")
session.use_role("ACCOUNTADMIN")
df = session.table("member_info_v")

## Feature Store for Feature Engineering
In this notebook, we created our member and program features using Snowflake's Feature Store. No data was moved outside of Snowflake and governance was maintained through centralized feature management.<br>
**Key Benefits Achieved:**
- **Centralized Feature Management**: Features are stored and versioned in Snowflake's Feature Store
- **Feature Reusability**: Features can be shared across multiple models and teams
- **Online Serving**: Features can be served in real-time for inference
- **Governance**: Complete feature lineage and usage tracking
- **Point-in-Time Correctness**: Features are retrieved as they existed at specific timestamps



#### Create Feature Store for Members

In [14]:
id_col = ["ALT_PRSN_ID"]

numeric_cols = [
    "AGE",
    "BEHAVIORAL_STRAIN",
    "FINANCIAL_STRAIN",
    "MEDICAL_STRAIN",
    "GEOGRAPHICAL_STRAIN",
    "HIGH_COST_CLAIMANT",
    "HOUSE_DYNAMICS",
    "NEED_A_HAND",
    "RISKY_BEHAVIOR_COUNT",
    "ERG_ACTUARIAL_RISK_SCORE",
    "ERG_PROSPECTIVE_RISK_SCORE",
    "ERG_DEMOGRAPHIC_RISK_SCORE",
    "HCC_DEMO_RISK_SCORE",
    "HCC_CHRONIC_RISK_SCORE",
    "HCC_ACUTE_RISK_SCORE",
    "TOTAL_HCC_RISK_SCORE",
    "DAYS_WITH_PROVIDER",
    "TOTAL_PAID_YEAR",
    "FACILITY_PAID_YEAR",
    "INPATIENT_PAID_YEAR",
    "OUTPATIENT_ER_PAID_YEAR",
    "OUTPATIENT_NON_ER_PAID_YEAR",
    "PROFESSIONAL_PAID_YEAR",
    "PRESCRIPTIONS_PAID_YEAR",
    "NUMBER_OF_PCP_VISITS",
    "NUMBER_OF_SPECIALIST_VISITS",
    "NUMBER_OF_CHIROPRACTOR_VISITS",
    "NUMBER_OF_ER_VISITS",
    "NUMBER_OF_IP_ADMITS",
]

categorical_cols = [
    "GENDER",
    "PHENOTYPE_NAME",
    "STRAIN_LEVEL",
    "INDIVIDUAL_OR_GROUP_MEDICAL_COVERAGE",
    "SHP_MEDICAL_COVERAGE",
    "MA_MEDICAL_COVERAGE",
    "FEP_COVERAGE",
]

fs = FeatureStore(
    session=session,
    database=session.get_current_database(),
    name=session.get_current_schema(),
    default_warehouse=session.get_current_warehouse(),
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

# Define entities
member_entity = Entity(
    name="member",
    join_keys=["ALT_PRSN_ID"],
    desc="Healthcare member entity"
)
fs.register_entity(member_entity)

# Create base features DataFrame
base_df = session.table("member_info_v")
base_df = base_df.filter(F.col("LATEST_ROW_FOR_MEMBER_FLAG") == 1)
base_df = base_df.select(id_col + numeric_cols + categorical_cols)

# Create age bins
base_df = base_df.with_columns(
    ["AGE_UNDER_18", "AGE_OVER_65"],
    [F.iff(F.col("AGE") < 18, 1, 0), F.iff(F.col("AGE") >= 65, 1, 0)],
)
binned_cols = [c + "_BIN" for c in numeric_cols]


# Bin numeric variables
est = snowml.KBinsDiscretizer(
    n_bins=5, encode="ordinal", input_cols=numeric_cols, output_cols=binned_cols
)
base_df = est.fit(base_df).transform(base_df)

# Drop original numeric columns
base_df = base_df.drop(numeric_cols)

# Create conditions DataFrame
conditions_df = session.table("member_hcc_condition_v")

# Create care gaps DataFrame
care_gaps_df = session.table("member_care_gap_v")

# Create feature views using FeatureView class
base_fv = feature_view.FeatureView(
    name="member_base_features",
    entities=[member_entity],
    feature_df=base_df,
    desc="Base member demographic and health features"
)

conditions_fv = feature_view.FeatureView(
    name="member_conditions",
    entities=[member_entity],
    feature_df=conditions_df,
    desc="Member health conditions"
)

care_gaps_fv = feature_view.FeatureView(
    name="member_care_gaps",
    entities=[member_entity],
    feature_df=care_gaps_df,
    desc="Member care gaps"
)

# Register feature views
registered_base_fv = fs.register_feature_view(base_fv, "v1", overwrite=True)
registered_conditions_fv = fs.register_feature_view(
    conditions_fv, "v1", overwrite=True)
registered_care_gaps_fv = fs.register_feature_view(
    care_gaps_fv, "v1", overwrite=True)

#### Create Feature Store for Programs

In [15]:
program_cols = [
    "PROGRAM_TIER",
    "PROGRAM_DELIVERY_MAIL",
    "PROGRAM_DELIVERY_EMAIL",
    "PROGRAM_DELIVERY_CALL",
    "PROGRAM_DELIVERY_MESSAGE",
    "PROGRAM_INTERVENTION_LEVEL",
]


fs = FeatureStore(
    session=session,
    database=session.get_current_database(),
    name=session.get_current_schema(),
    default_warehouse=session.get_current_warehouse(),
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

# Define program entity
program_entity = Entity(
    name="program",
    join_keys=["PROGRAM_ID"],
    desc="Healthcare program entity"
)
fs.register_entity(program_entity)

# Create program features DataFrame
program_df = (
    session.table("syn_member_program_v")
    .select(["PROGRAM_ID", "PROGRAM_ESTIMATED_SAVINGS_MEMBER"] + program_cols)
    .distinct()
)

# Create feature string, required format for the recommendation engine
program_cols_s = [c + "_STR" for c in program_cols]
program_df = program_df.with_columns(
    program_cols_s, [F.concat(F.lit(c), F.lit(":"), F.col(c))
                     for c in program_cols]
).with_column("FEATURE_STRING", F.concat_ws(F.lit(" "), *program_cols_s))

# Add program weighting: max-scaled estimated cost savings
max_savings = program_df.select(F.max(F.col("PROGRAM_ESTIMATED_SAVINGS_MEMBER"))).collect()[
    0
][0]
program_df = program_df.with_column(
    "PROGRAM_WEIGHT", F.col("PROGRAM_ESTIMATED_SAVINGS_MEMBER") / max_savings
)

# Create feature view using FeatureView class
program_fv = feature_view.FeatureView(
    name="program_features",
    entities=[program_entity],
    feature_df=program_df,
    desc="Program features for recommendation engine"
)

# Register feature view
registered_program_fv = fs.register_feature_view(
    program_fv, "v1", overwrite=True)

Helper Functions for using feature store

In [16]:
def build_program_features(session: Session, table_name: str) -> str:

    # Set up Feature Store
    fs = FeatureStore(
        session=session,
        database=session.get_current_database(),
        name=session.get_current_schema(),
        default_warehouse=session.get_current_warehouse(),
        creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
    )
    # Read features from feature view
    program_features = fs.read_feature_view(registered_program_fv)

    # Save final feature table
    program_features.select("PROGRAM_ID", "FEATURE_STRING", "PROGRAM_WEIGHT").write.mode(
        "overwrite"
    ).save_as_table(table_name)

    return program_features


def build_member_features(session: Session, table_name: str) -> str:
    # Set up Feature Store
    fs = FeatureStore(
        session=session,
        database=session.get_current_database(),
        name=session.get_current_schema(),
        default_warehouse=session.get_current_warehouse(),
        creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
    )

    # Read features from feature views
    base_features = fs.read_feature_view(registered_base_fv)
    conditions_features = fs.read_feature_view(registered_conditions_fv)
    care_gaps_features = fs.read_feature_view(registered_care_gaps_fv)

    # Join all features
    df_transformed = base_features.join(
        conditions_features,
        on=["ALT_PRSN_ID"],
        how="left"
    ).join(
        care_gaps_features,
        on=["ALT_PRSN_ID"],
        how="left"
    )

    # Replace nulls with 0 for condition and care gap columns
    condition_cols = [
        col for col in conditions_features.columns if col != "ALT_PRSN_ID"]
    caregap_cols = [
        col for col in care_gaps_features.columns if col != "ALT_PRSN_ID"]
    df_transformed = df_transformed.fillna(0, condition_cols + caregap_cols)

    # Create feature string for recommendation engine
    member_cols = [
        col for col in df_transformed.columns if col != "ALT_PRSN_ID"]
    member_cols_s = [c + "_STR" for c in member_cols]
    df_transformed = df_transformed.with_columns(
        member_cols_s,
        [F.upper(F.concat(F.lit(c), F.lit(":"), F.col(c)))
         for c in member_cols],
    ).with_column("FEATURE_STRING", F.concat_ws(F.lit(" "), *member_cols_s))

    # Save final feature table
    df_transformed.select("ALT_PRSN_ID", "FEATURE_STRING").write.mode(
        "overwrite"
    ).save_as_table(table_name)
    return df_transformed

Call Feature Views to create Member Feature Table

In [17]:
# U
member_features_df = build_member_features(session, "COHORT_BUILDER.RAW.MEMBER_FEATURES_V2")

View the Member Feature Table


In [18]:
session.table("COHORT_BUILDER.RAW.MEMBER_FEATURES_V2").show()

----------------------------------------------------------------------
|"ALT_PRSN_ID"  |"FEATURE_STRING"                                    |
----------------------------------------------------------------------
|48360507       |AGE_BIN:1 BEHAVIORAL_STRAIN_BIN:3 FINANCIAL_STR...  |
|7364038        |AGE_BIN:1 BEHAVIORAL_STRAIN_BIN:4 FINANCIAL_STR...  |
|69281495       |AGE_BIN:1 BEHAVIORAL_STRAIN_BIN:3 FINANCIAL_STR...  |
|9228973        |AGE_BIN:4 BEHAVIORAL_STRAIN_BIN:3 FINANCIAL_STR...  |
|49530331       |AGE_BIN:1 BEHAVIORAL_STRAIN_BIN:3 FINANCIAL_STR...  |
|9022476        |AGE_BIN:1 BEHAVIORAL_STRAIN_BIN:3 FINANCIAL_STR...  |
|8212186        |AGE_BIN:2 BEHAVIORAL_STRAIN_BIN:4 FINANCIAL_STR...  |
|8710571        |AGE_BIN:2 BEHAVIORAL_STRAIN_BIN:4 FINANCIAL_STR...  |
|48345061       |AGE_BIN:3 BEHAVIORAL_STRAIN_BIN:3 FINANCIAL_STR...  |
|9142961        |AGE_BIN:2 BEHAVIORAL_STRAIN_BIN:4 FINANCIAL_STR...  |
----------------------------------------------------------------------



In [19]:
program_table_name = "PROGRAM_FEATURES"
# build_program_features(session, program_table_name)
session.table(program_table_name).show()

---------------------------------------------------------------------------------------------
|"PROGRAM_ID"  |"FEATURE_STRING"                                    |"PROGRAM_WEIGHT"       |
---------------------------------------------------------------------------------------------
|10010         |PROGRAM_TIER:ELIGIBILITY PROGRAM_DELIVERY_MAIL:...  |0.0017142857142857142  |
|10081         |PROGRAM_TIER:CARE_GAP PROGRAM_DELIVERY_MAIL:0 P...  |0.05714285714285714    |
|10048         |PROGRAM_TIER:CARE_GAP PROGRAM_DELIVERY_MAIL:0 P...  |0.001142857142857143   |
|10088         |PROGRAM_TIER:CLINICAL PROGRAM_DELIVERY_MAIL:0 P...  |0.05714285714285714    |
|10070         |PROGRAM_TIER:CARE_GAP PROGRAM_DELIVERY_MAIL:0 P...  |0.01                   |
|10013         |PROGRAM_TIER:CARE_GAP PROGRAM_DELIVERY_MAIL:0 P...  |0.0010285714285714286  |
|10064         |PROGRAM_TIER:CLINICAL PROGRAM_DELIVERY_MAIL:0 P...  |0.008571428571428572   |
|10036         |PROGRAM_TIER:CARE_GAP PROGRAM_DELIVERY_MAIL:

## Modern ML Training with Snowflake ML Jobs

This notebook demonstrates how to use **Snowflake ML Jobs** to train machine learning models on Snowflake's compute infrastructure. This approach provides several advantages over traditional stored procedures:

### **Key Benefits of ML Jobs:**

- **Distributed Training**: Run training on multiple compute nodes for faster processing with Distributed Modeling Classess and Ray
- **Development Environment**: Use your preferred IDE (VS Code, Cursor) while leveraging Snowflake's compute
- **Custom Dependencies**: Install and use custom Python packages within the runtime environment
- **Monitoring**: Track job progress, debug issues, and monitor execution through Snowflake's APIs

### **ML Jobs vs Stored Procedures:**

| Feature | ML Jobs | Stored Procedures |
|---------|---------|-------------------|
| **Dependencies** | Custom Python packages | Limited package support |
| **Scaling** | Multi-node distributed training on CPUs/GPUs| Warehouse execution |
| **Monitoring** | Rich job management APIs | Basic execution tracking |


### **How It Works:**

1. **Function Decoration**: The `@remote` decorator marks the function for execution on Snowflake compute
2. **Job Submission**: Function invocation returns an `MLJob` object for monitoring
3. **Distributed Execution**: Training runs on specified compute pool with multiple instances
4. **Artifact Storage**: Model and artifacts are automatically saved to Snowflake stages
5. **Result Retrieval**: Use `job.result()` to get the return value after completion


## Create a Compute Pool to run ML Training Job

In [ ]:
def create_compute_pool(name: str, instance_family: str, min_nodes: int = 1, max_nodes: int = 10):
    query = f"""
        CREATE COMPUTE POOL IF NOT EXISTS {name}
            MIN_NODES = {min_nodes}
            MAX_NODES = {max_nodes}
            INSTANCE_FAMILY = {instance_family}
    """
    return session.sql(query).collect()


compute_pool = "DEMO_POOL_CPU"
create_compute_pool(compute_pool, "CPU_X64_S", 1, 5)

In [ ]:
# LightGBM-based ML Jobs implementation 
from snowflake.ml.jobs import remote
from snowflake.snowpark import Session
import joblib
import os
import json
import pickle
from time import perf_counter

# Set up compute pool for ML Jobs
compute_pool = "DEMO_POOL_CPU"  # Replace with your compute pool name
stage_name = "payload_stage"    # Replace with your stage name

# Define custom dependencies for ML Jobs
pip_requirements = [
    "numpy==1.26.4",
    "scikit-learn",
    "lightgbm==4.5.0",
    "pandas",
]


@remote(compute_pool, stage_name=stage_name, target_instances=1, pip_requirements=pip_requirements)
def train_recommender(session: Session, model_name: str, save_mode: str = "registry", output_dir: str = None) -> str:
    import lightgbm as lgb
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score, precision_score, recall_score
    from snowflake.ml.registry import Registry
    import pandas as pd
    import numpy as np
    import joblib
    import os
    import json
    import pickle
    from time import perf_counter

    start = perf_counter()

    # Member features DataFrame - limit to 1000 members for testing
    df_member = session.table("member_features").limit(1000)

    # Get the member IDs from the limited member set
    member_ids = df_member.select("ALT_PRSN_ID").to_pandas()[
        "ALT_PRSN_ID"].tolist()

    # Program to members DataFrame - only include records for our limited members
    df_program = session.table("syn_member_program_v").filter(
        F.col("ALT_PRSN_ID").isin(member_ids)
    )

    # Get the program IDs from the filtered program set
    program_ids = df_program.select("PROGRAM_ID").distinct().to_pandas()[
        "PROGRAM_ID"].tolist()

    # Program features DataFrame - only include features for programs in our subset
    df_program_feature = session.table("program_features").filter(
        F.col("PROGRAM_ID").isin(program_ids)
    )

    # Pull Snowpark DataFrames into Pandas DataFrames
    member_df = df_member.to_pandas()
    program_df = df_program.to_pandas()
    program_feature_df = df_program_feature.to_pandas()

    print(
        f"Training with {len(member_df)} members and {len(program_ids)} programs")

    # Create member-program suitability matrix
    # This will be our target variable (1 if member is in program, 0 otherwise)
    suitability_matrix = np.zeros((len(member_df), len(program_ids)))

    # Create member ID to index mapping
    member_id_to_idx = {member_id: idx for idx,
                        member_id in enumerate(member_df["ALT_PRSN_ID"])}
    program_id_to_idx = {program_id: idx for idx,
                         program_id in enumerate(program_ids)}

    # Fill suitability matrix based on actual program enrollments
    for _, row in program_df.iterrows():
        member_idx = member_id_to_idx.get(row["ALT_PRSN_ID"])
        program_idx = program_id_to_idx.get(row["PROGRAM_ID"])
        if member_idx is not None and program_idx is not None:
            suitability_matrix[member_idx, program_idx] = 1

    # Prepare member features for training
    # Parse feature strings into individual features
    member_features_list = []
    for feature_string in member_df["FEATURE_STRING"]:
        features = {}
        for feature in feature_string.split():
            if ":" in feature:
                key, value = feature.split(":", 1)
                features[key] = value
        member_features_list.append(features)

    # Convert to DataFrame
    member_features_df = pd.DataFrame(member_features_list).fillna(0)

    # Convert categorical features to numeric
    for col in member_features_df.columns:
        if member_features_df[col].dtype == 'object':
            member_features_df[col] = pd.Categorical(
                member_features_df[col]).codes

    # Prepare program features for each program
    program_features_list = []
    for program_id in program_ids:
        program_row = program_feature_df[program_feature_df["PROGRAM_ID"] == program_id]
        if not program_row.empty:
            features = {}
            for feature in program_row.iloc[0]["FEATURE_STRING"].split():
                if ":" in feature:
                    key, value = feature.split(":", 1)
                    features[key] = value
            program_features_list.append(features)
        else:
            program_features_list.append({})

    # Convert program features to DataFrame
    program_features_df = pd.DataFrame(program_features_list).fillna(0)

    # Convert categorical features to numeric
    for col in program_features_df.columns:
        if program_features_df[col].dtype == 'object':
            program_features_df[col] = pd.Categorical(
                program_features_df[col]).codes

    # Create combined features (member features + program features for each program)
    # This creates a feature matrix where each row is a member-program pair
    X_train = []
    y_train = []

    for member_idx, member_id in enumerate(member_df["ALT_PRSN_ID"]):
        member_features = member_features_df.iloc[member_idx].values

        for program_idx, program_id in enumerate(program_ids):
            program_features = program_features_df.iloc[program_idx].values

            # Combine member and program features
            combined_features = np.concatenate(
                [member_features, program_features])
            X_train.append(combined_features)

            # Target: 1 if member is in program, 0 otherwise
            y_train.append(suitability_matrix[member_idx, program_idx])

    X_train = np.array(X_train)
    y_train = np.array(y_train)

    print(f"Training data shape: {X_train.shape}")
    print(
        f"Positive samples: {np.sum(y_train)} / {len(y_train)} ({np.sum(y_train)/len(y_train)*100:.2f}%)")

    # Split data for training and validation
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
    )

    # Train LightGBM model
    model = lgb.LGBMClassifier(
        objective='binary',
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        verbose=-1
    )

    model.fit(X_train_split, y_train_split)

    # Calculate metrics
    train_pred = model.predict(X_train_split)
    train_pred_proba = model.predict_proba(X_train_split)[:, 1]
    val_pred = model.predict(X_val_split)
    val_pred_proba = model.predict_proba(X_val_split)[:, 1]

    metrics = {
        "train_auc": float(roc_auc_score(y_train_split, train_pred_proba)),
        "val_auc": float(roc_auc_score(y_val_split, val_pred_proba)),
        "train_precision": float(precision_score(y_train_split, train_pred, zero_division=0)),
        "val_precision": float(precision_score(y_val_split, val_pred, zero_division=0)),
        "train_recall": float(recall_score(y_train_split, train_pred, zero_division=0)),
        "val_recall": float(recall_score(y_val_split, val_pred, zero_division=0)),
        "num_members": len(member_df),
        "num_programs": len(program_ids),
        "num_interactions": int(np.sum(suitability_matrix)),
        "training_time_seconds": perf_counter() - start
    }

    # Create model artifacts dictionary
    model_artifacts = {
        "model": model,
        "member_id_to_idx": member_id_to_idx,
        "program_id_to_idx": program_id_to_idx,
        "member_features_df": member_features_df,
        "program_features_df": program_features_df,
        "program_ids": program_ids,
        "member_ids": member_df["ALT_PRSN_ID"].tolist()
    }
    # Ensure your X_train is a pandas DataFrame (not numpy)
    if not isinstance(X_train, pd.DataFrame):
        X_train = pd.DataFrame(X_train)

    # Use first few rows as sample input data
    sample_input_data = X_train.head(5).copy()

    # Optional: ensure all float dtypes (Snowflake prefers that)
    sample_input_data = sample_input_data.astype(float)

    if save_mode == "local":
        # Save model locally
        print("Saving model to disk...", end="")
        output_dir = output_dir or os.path.dirname(__file__)
        model_subdir = os.environ.get("SNOWFLAKE_SERVICE_NAME", "output")
        model_dir = os.path.join(output_dir, model_subdir) if not output_dir.endswith(
            model_subdir) else output_dir
        os.makedirs(model_dir, exist_ok=True)

        # Save main model
        with open(os.path.join(model_dir, "model.pkl"), "wb") as f:
            pickle.dump(model, f)

        # Save model artifacts
        with open(os.path.join(model_dir, "model_artifacts.pkl"), "wb") as f:
            pickle.dump(model_artifacts, f)

        # Save metrics
        with open(os.path.join(model_dir, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

        print("✓")

    elif save_mode == "registry":
        # Save model to registry
        print("Logging model to Model Registry...", end="")

        registry = Registry(session)

        # Save to registry
        registry.log_model(
            model=model,
            model_name=model_name,
            metrics=metrics,
            target_platforms=["SNOWPARK_CONTAINER_SERVICES", "WAREHOUSE"]
            sample_input_data=sample_input_data,
            conda_dependencies=["numpy==1.26.4",
                              "scikit-learn", "lightgbm==4.5.0", "pandas"]
        )
    elapsed = perf_counter() - start
    print(f"Training completed in {elapsed:.2f} seconds")
    print(f"Model metrics: {metrics}")

    return "SUCCESS"

In [26]:
# Submit the LightGBM ML Job for training
model_name = "lightgbm_recommendation_model_v1"
save_mode = "registry"  # Options: "registry" or "local"

# For local saving, specify output directory
# save_mode = "local"
# output_dir = "/path/to/output"

job = train_recommender(session, model_name, save_mode=save_mode)

print(f"Job ID: {job.id}")
print(f"Job Status: {job.status}")

# Wait for job completion and get logs
job.wait()
print("Job completed!")
job.show_logs()

Job ID: COHORT_BUILDER.RAW.TRAIN_RECOMMENDER_RCELN4WWD578
Job Status: PENDING
Job completed!
Training with 1000 members and 77 programs
Training data shape: (77000, 259)
Positive samples: 5422.0 / 77000 (7.04%)
2025-10-28 13:44:06,192 - INFO - Using non-live commit model version
2025-10-28 13:44:06,405 - INFO - Logging the model on Container Runtime for ML without specifying `target_platforms`. Default to `target_platforms=["SNOWPARK_CONTAINER_SERVICES"]`.
2025-10-28 13:44:06,405 - INFO - Setting `relax_version=False` as this model will run in Snowpark Container Services or in Warehouse with a specified artifact_repository_map where exact version  specifications will be honored.
2025-10-28 13:44:06,405 - INFO - Start packaging and uploading your model. It might take some time based on the size of the model.
2025-10-28 13:44:14,935 - INFO - Snowflake Connector for Python Version: 3.17.3, Python Version: 3.10.18, Platform: Linux-5.15.185-14.2025090508g6b8d30a+snow+aws+5.15+amd64.x86_64-x

## Load Model from Registry and Run Container Inference

Following the modern ML deployment pattern, we'll:
1. **Retrieve the model** from Model Registry using the default version
2. **Run inference** using container runtime with `model.run()`
3. **Generate recommendations** for all members efficiently


In [29]:
# Initialize Model Registry and retrieve model
model_name = "LIGHTGBM_RECOMMENDATION_MODEL_V1"
registry = Registry(session)

# Get the default model version for container inference
mv_base = registry.get_model(model_name).default
metrics = mv_base.show_metrics()
print(f"Retrieved model: {mv_base}")
print(f"Model metrics: {metrics}")

# Check what services are available
print("\nChecking available services...")
try:
    services = session.sql("SHOW SERVICES").collect()
    print("Available services:")
    for service in services:
        print(f"  - {service['name']}")
except Exception as e:
    print(f"Error checking services: {e}")



Retrieved model: ModelVersion(
  name='LIGHTGBM_RECOMMENDATION_MODEL_V1',
  version='GIANT_MOLE_3',
)
Model metrics: {'train_auc': 0.9411813822076776, 'val_auc': 0.9306943357572721, 'train_precision': 0.7147862648913805, 'val_precision': 0.6657894736842105, 'train_recall': 0.2351313969571231, 'val_recall': 0.2333948339483395, 'num_members': 1000, 'num_programs': 77, 'num_interactions': 5422, 'training_time_seconds': 898.0840659580135}

Checking available services...
Available services:
  - LIGHTGBM_RECOMMENDATION_MODEL_V1_GIANT_MOLE_3_SERVICE
  - LIGHTGBM_RECOMMENDATION_MODEL_V1_HEAVY_PENGUIN_4_SERVICE
  - MODEL_BUILD_6096E110
  - MODEL_BUILD_6F54B9EC
  - TRAIN_RECOMMENDER_1510U37Z8J7JZ
  - TRAIN_RECOMMENDER_15TP1NNS3431J
  - TRAIN_RECOMMENDER_1DI394RCA1VRV
  - TRAIN_RECOMMENDER_1E8PSV8LBKU2G
  - TRAIN_RECOMMENDER_1GI7XLB1D3A2X
  - TRAIN_RECOMMENDER_1GNLF26GDYJDD
  - TRAIN_RECOMMENDER_1J5AY6WJ1KQEL
  - TRAIN_RECOMMENDER_1KYXQMOVFTNPY
  - TRAIN_RECOMMENDER_1QQQVD80BZFU5
  - TRAIN_RECOMM

In [ ]:
DROP SERVICE IF EXISTS LIGHTGBM_RECOMMENDATION_MODEL_V1_GIANT_MOLE_3_SERVICE;

In [67]:
mv_base.create_service(
    service_name="LIGHTGBM_RECOMMENDATION_MODEL_SERVICE_V2",
    service_compute_pool="DEMO_POOL_CPU",
    ingress_enabled=True,
    max_instances=1
)

create_service logs saved to: /Users/kwells/Library/Logs/snowflake-ml/model_deploy_94402b22_1761669286.log        
To see logs in console, set log level to INFO: logging.getLogger().setLevel(logging.INFO)                     
Creating model inference service: starting model image build...:  67%|██████▋   | 4/6 [00:00<00:00,  6.29it/s]

Model service deployment failed: Timeout waiting for service MODEL_BUILD_A61FF323 to reach status RUNNING after 30 minutes


❌ ERROR: Model service deployment failed: Timeout waiting for service MODEL_BUILD_A61FF323 to reach status RUNNING after 30 minutes:  67%|██████▋   | 4/6 [30:02<15:01, 450.60s/it]


RuntimeError: Model service deployment failed: Timeout waiting for service MODEL_BUILD_A61FF323 to reach status RUNNING after 30 minutes

SyntaxError: invalid syntax (3007686784.py, line 1)

### Run Container Inference

In [46]:
# Member features DataFrame - limit to 1000 members for testing
df_member = session.table("member_features").limit(2000)


# Get the member IDs from the limited member set
member_ids = df_member.select("ALT_PRSN_ID").to_pandas()[
    "ALT_PRSN_ID"].tolist()


# Program to members DataFrame - only include records for our limited members
df_program = session.table("syn_member_program_v").filter(
    F.col("ALT_PRSN_ID").isin(member_ids)
)

# Get the program IDs from the filtered program set
program_ids = df_program.select("PROGRAM_ID").distinct().to_pandas()[
    "PROGRAM_ID"].tolist()


# Program features DataFrame - only include features for programs in our subset
df_program_feature = session.table("program_features").filter(
    F.col("PROGRAM_ID").isin(program_ids)
)

df_member_pd = df_member.to_pandas()
df_program_pd = df_program.to_pandas()
df_program_feature_pd = df_program_feature.to_pandas()



# Prepare member features for training
# Parse feature strings into individual features
member_features_list = []
for feature_string in df_member_pd["FEATURE_STRING"]:
    features = {}
    for feature in feature_string.split():
        if ":" in feature:
            key, value = feature.split(":", 1)
            features[key] = value
    member_features_list.append(features)

# # Convert to DataFrame
member_features_df = pd.DataFrame(member_features_list).fillna(0)

# Convert categorical features to numeric
for col in member_features_df.columns:
    if member_features_df[col].dtype == 'object':
        member_features_df[col] = pd.Categorical(member_features_df[col]).codes

# Prepare program features for each program
program_features_list = []
for program_id in program_ids:
    program_row = df_program_feature_pd[df_program_feature_pd["PROGRAM_ID"] == program_id]
    if not program_row.empty:
        features = {}
        for feature in program_row.iloc[0]["FEATURE_STRING"].split():
            if ":" in feature:
                key, value = feature.split(":", 1)
                features[key] = value
        program_features_list.append(features)
    else:
        program_features_list.append({})

# Convert program features to DataFrame
program_features_df = pd.DataFrame(program_features_list).fillna(0)

# Convert categorical features to numeric
for col in program_features_df.columns:
    if program_features_df[col].dtype == 'object':
        program_features_df[col] = pd.Categorical(
            program_features_df[col]).codes

# # Create combined features (member features + program features for each program)
# # This creates a feature matrix where each row is a member-program pair
inference_data = []

for member_idx, member_id in enumerate(df_member_pd["ALT_PRSN_ID"]):
    member_features = member_features_df.iloc[member_idx].values

    for program_idx, program_id in enumerate(program_ids):
        program_features = program_features_df.iloc[program_idx].values

        # Combine member and program features
        combined_features = np.concatenate([member_features, program_features])
        inference_data.append(combined_features)


inference_data = np.array(inference_data)

In [ ]:
# Create a DataFrame with the feature columns that the model expects (147 features)
feature_columns = [
    f"INPUT_FEATURE_{i}" for i in range(inference_data.shape[1])]
inference_df = pd.DataFrame(inference_data, columns=feature_columns)

# Add metadata columns for tracking
inference_df["ALT_PRSN_ID"] = [df_member_pd["ALT_PRSN_ID"].iloc[i //
                                                                len(program_ids)] for i in range(len(inference_data))]
inference_df["PROGRAM_ID"] = [program_ids[i %
                                          len(program_ids)] for i in range(len(inference_data))]

# Convert to Snowpark DataFrame
inference_snowpark = session.create_dataframe(inference_df)

print(f"Created inference DataFrame with {inference_snowpark.count()} rows and {len(feature_columns)} features")
print("Sample of inference data:")
inference_snowpark.select("ALT_PRSN_ID", "PROGRAM_ID", "INPUT_FEATURE_0", "INPUT_FEATURE_1", "INPUT_FEATURE_2").show(5)

# Run inference using the correct service
try:
    print("Running inference with container service...")
    service_name="LIGHTGBM_RECOMMENDATION_MODEL_SERVICE",
    predictions = mv_base.run(inference_snowpark, function_name="predict", service_name='LIGHTGBM_RECOMMENDATION_MODEL_SERVICE').rename(
        '"output_feature_0"', 'RECOMMENDATION_SCORE')
    print("✅ Container inference successful!")
    
except Exception as e:
    print(f"Container inference failed: {e}")
    print("Trying without explicit service name...")
    

# Show predictions
print("✅ Inference completed successfully!")
predictions.show()

Created inference DataFrame with 168000 rows and 259 features
Sample of inference data:
--------------------------------------------------------------------------------------------
|"ALT_PRSN_ID"  |"PROGRAM_ID"  |"INPUT_FEATURE_0"  |"INPUT_FEATURE_1"  |"INPUT_FEATURE_2"  |
--------------------------------------------------------------------------------------------
|01121569       |10037         |0                  |0                  |2                  |
|01121569       |10051         |0                  |0                  |2                  |
|01121569       |10022         |0                  |0                  |2                  |
|01121569       |10014         |0                  |0                  |2                  |
|01121569       |10034         |0                  |0                  |2                  |
--------------------------------------------------------------------------------------------

Running inference with container service...
✅ Container inference successf

SnowparkSQLException: (1304): 01c0037a-030f-8ead-0003-876b035ce372: 395004 (55000): Service LIGHTGBM_RECOMMENDATION_MODEL_V1_GIANT_MOLE_3_SERVICE not reachable: service failed or deleted.

In [ ]:
# Get top 3 recommendations per member using window functions
from snowflake.snowpark import Window

# Create a window partitioned by member, ordered by score descending
window_spec = Window.partition_by("ALT_PRSN_ID").order_by(
    F.col("RECOMMENDATION_SCORE").desc())

# Add rank column
ranked_predictions = predictions.with_column(
    "RANK", F.row_number().over(window_spec))

# Filter to top 3 per member
top_3_recommendations = ranked_predictions.filter(F.col("RANK") <= 3)

# Show results
print("Top 3 recommendations per member:")
top_3_recommendations.select(
    "ALT_PRSN_ID", "PROGRAM_ID", "RECOMMENDATION_SCORE", "RANK").show(15)

# Save recommendations to tables
print("\nSaving recommendations to Snowflake tables...")
predictions.write.mode("overwrite").save_as_table("LIGHTGBM_PROGRAM_RECOMMENDATIONS")
top_3_recommendations.write.mode("overwrite").save_as_table("LIGHTGBM_TOP3_RECOMMENDATIONS")
print("✅ Saved all predictions to LIGHTGBM_PROGRAM_RECOMMENDATIONS table")
print("✅ Saved top 3 recommendations to LIGHTGBM_TOP3_RECOMMENDATIONS table")


## Get Top Recommendations per Member

Now we can easily get the top 3 program recommendations for each member using Snowpark DataFrame operations.


In [ ]:
# Get top 3 recommendations per member using window functions
from snowflake.snowpark import Window

# Create a window partitioned by member, ordered by score descending
window_spec = Window.partition_by("ALT_PRSN_ID").order_by(
    F.col("RECOMMENDATION_SCORE").desc())

# Add rank column
ranked_predictions = predictions.with_column(
    "RANK", F.row_number().over(window_spec))

# Filter to top 3 per member
top_3_recommendations = ranked_predictions.filter(F.col("RANK") <= 3)

# Show results
print("Top 3 recommendations per member:")
top_3_recommendations.select(
    "ALT_PRSN_ID", "PROGRAM_NAME", "RECOMMENDATION_SCORE", "RANK").show(15)